In [ ]:
!pip uninstall -y f5-tts -q
!pip install -q git+https://github.com/AI4Bharat/IndicF5.git
!pip install -q huggingface_hub soundfile librosa
!pip -q install -U "nemo_toolkit[asr]"

  Preparing metadata (setup.py) ... done


In [ ]:
from huggingface_hub import login

login()

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Install necessary libraries
!pip install -q fastapi uvicorn python-multipart pyngrok

import json
import torch
import soundfile as sf
from pathlib import Path
from huggingface_hub import snapshot_download
from IPython.display import Audio, display

# For FastAPI and ngrok
import uvicorn
from fastapi import FastAPI, UploadFile, File, Form
from pydantic import BaseModel
from pyngrok import ngrok
import nest_asyncio
import threading
import time
import os
import tempfile
from starlette.responses import StreamingResponse
from io import BytesIO
from fastapi.middleware.cors import CORSMiddleware


# ==================================================================
# 1. Load TTS Model (from cell 4EwA0SD-u1yd)
# ==================================================================
print("Loading TTS model...")
from f5_tts.model import CFM, DiT
from f5_tts.model.utils import get_tokenizer
from f5_tts.infer.utils_infer import (
    load_vocoder,
    preprocess_ref_audio_text,
    infer_process,
)

MODEL_ID_TTS = "tryorato/orato-tts-hindi-v1"
DEVICE_TTS = "cuda" if torch.cuda.is_available() else "cpu"

snap_tts = Path(snapshot_download(MODEL_ID_TTS))
voices_tts = json.loads((snap_tts / "voices.json").read_text(encoding="utf-8"))

# Load a default voice for model initialization
default_voice = "female"
ref_wav_tts = snap_tts / voices_tts[default_voice]["wav"]
ref_text_tts = voices_tts[default_voice]["ref_text"]

ckpt_path_tts = snap_tts / "model.pt"
vocab_path_tts = snap_tts / "vocab.txt"

vocab_char_map_tts, vocab_size_tts = get_tokenizer(
    str(vocab_path_tts),
    "custom"
)

model_tts = CFM(
    transformer=DiT(
        dim=1024,
        depth=22,
        heads=16,
        ff_mult=2,
        text_dim=512,
        conv_layers=4,
        text_num_embeds=vocab_size_tts,
        mel_dim=100,
    ),
    mel_spec_kwargs={
        "n_fft": 1024,
        "hop_length": 256,
        "win_length": 1024,
        "n_mel_channels": 100,
        "target_sample_rate": 24000,
        "mel_spec_type": "vocos",
    },
    odeint_kwargs={
        "method": "euler",
    },
    vocab_char_map=vocab_char_map_tts,
)

checkpoint_tts = torch.load(
    ckpt_path_tts,
    map_location="cpu",
    weights_only=False,
)

model_tts.load_state_dict(
    checkpoint_tts["model_state_dict"],
    strict=True,
)

model_tts = model_tts.to(DEVICE_TTS)
model_tts.eval()

vocoder_tts = load_vocoder(
    vocoder_name="vocos",
    is_local=False,
    device=DEVICE_TTS,
)
print("TTS model loaded.")

# ==================================================================
# 2. Load LLM Model (from cell raiC_eh8zdjk)
# ==================================================================
print("Loading LLM model...")
from transformers import pipeline

model_id_llm = "meta-llama/Llama-3.2-1B-Instruct"
pipe_llm = pipeline(
    "text-generation",
    model=model_id_llm,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print("LLM model loaded.")

# ==================================================================
# 3. Load ASR Model (from cell hmVEl0lc4f1u)
# ==================================================================
print("Loading ASR model...")
import nemo.collections.asr as nemo_asr

STT_MODEL_ASR = "nvidia/stt_en_fastconformer_ctc_large"

stt_model_asr = nemo_asr.models.ASRModel.from_pretrained(STT_MODEL_ASR)
stt_model_asr = stt_model_asr.to(DEVICE_TTS) # Modified to use detected device
stt_model_asr.eval()
print("ASR model loaded.")

Loading TTS model...


Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
DEBUG:jieba:Loading model from cache /tmp/jieba.cache
Loading model cost 4.203 seconds.
DEBUG:jieba:Loading model cost 4.203 seconds.
Prefix dict has been built successfully.
DEBUG:jieba:Prefix dict has been built successfully.


Word segmentation module jieba initialized.



Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

LLM model loaded.
Loading ASR model...


[NeMo I 2026-08-10 16:15:28 mixins:194] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-08-10 16:15:28 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    batch_size: 1
    shuffle: true
    num_workers: 8
    pin_memory: true
    use_start_end_token: false
    trim_silence: false
    max_duration: 20
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: fully_randomized
    bucketing_batch_size: null
    
[NeMo W 2026-08-10 16:15:28 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath: null
    sample_rate: 16000
    batch_size: 32
    shuffle: false
    num_workers: 8
    pin_m

[NeMo I 2026-08-10 16:15:30 save_restore_connector:287] Model EncDecCTCModelBPE was successfully restored from /root/.cache/huggingface/hub/models--nvidia--stt_en_fastconformer_ctc_large/snapshots/5a84a7a3bee8d9bd414c6719ddfea7bc723e3961/stt_en_fastconformer_ctc_large.nemo.
ASR model loaded.


In [ ]:
# ==================================================================
# Imports
# ==================================================================

import os
import time
import tempfile
import threading
from io import BytesIO

import nest_asyncio
import uvicorn
import torch
import soundfile as sf

from fastapi import (
    FastAPI,
    UploadFile,
    File,
)
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import (
    StreamingResponse,
    FileResponse,
)
from pydantic import BaseModel

from pyngrok import ngrok


# ==================================================================
# FastAPI Application
# ==================================================================

app = FastAPI(
    title="Hindi AI Voice Assistant API",
    version="1.0.0",
)


# ==================================================================
# CORS
# ==================================================================
# Not required when frontend is served by this same FastAPI server,
# but kept for development if you later use Live Server again.
# ==================================================================

app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://127.0.0.1:5500",
        "http://localhost:5500",
    ],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)


# ==================================================================
# Frontend
# ==================================================================

FRONTEND_FILE = "/content/index.html"


@app.get("/")
async def frontend():
    return FileResponse(
        FRONTEND_FILE,
        media_type="text/html",
    )


# ==================================================================
# Request Models
# ==================================================================

class TTSRequest(BaseModel):
    text: str
    voice: str = "female"


class LLMRequest(BaseModel):
    prompt: str
    system_message: str = ""


# ==================================================================
# Status
# ==================================================================

@app.get("/status")
async def status():
    return {
        "status": "running"
    }


# ==================================================================
# TTS
# ==================================================================

@app.post("/tts")
async def tts_endpoint(request: TTSRequest):

    try:

        # ----------------------------------------------------------
        # Validate voice
        # ----------------------------------------------------------

        if request.voice not in voices_tts:

            return {
                "error": (
                    "Invalid voice selected. "
                    "Choose 'female' or 'male'."
                )
            }


        # ----------------------------------------------------------
        # Reference voice
        # ----------------------------------------------------------

        ref_wav_current = (
            snap_tts /
            voices_tts[request.voice]["wav"]
        )

        ref_text_current = (
            voices_tts[request.voice]["ref_text"]
        )


        # ----------------------------------------------------------
        # Preprocess reference audio
        # ----------------------------------------------------------

        ref_audio, ref_text_processed = (
            preprocess_ref_audio_text(
                str(ref_wav_current),
                ref_text_current,
            )
        )


        # ----------------------------------------------------------
        # TTS inference
        # ----------------------------------------------------------

        audio_output, sample_rate_output, _ = (
            infer_process(
                ref_audio=ref_audio,
                ref_text=ref_text_processed,
                gen_text=request.text,
                model_obj=model_tts,
                vocoder=vocoder_tts,
                mel_spec_type="vocos",
                nfe_step=32,
                cfg_strength=2.0,
                speed=1.0,
                device=DEVICE_TTS,
            )
        )


        # ----------------------------------------------------------
        # Convert audio to WAV in memory
        # ----------------------------------------------------------

        buffer = BytesIO()

        sf.write(
            buffer,
            audio_output,
            sample_rate_output,
            format="WAV",
        )

        buffer.seek(0)


        # ----------------------------------------------------------
        # Return audio
        # ----------------------------------------------------------

        return StreamingResponse(
            buffer,
            media_type="audio/wav",
        )


    except Exception as e:

        print(
            "TTS ERROR:",
            repr(e),
        )

        return {
            "error": str(e)
        }


# ==================================================================
# LLM
# ==================================================================

@app.post("/llm")
async def llm_endpoint(request: LLMRequest):

    try:

        # ----------------------------------------------------------
        # Build messages
        # ----------------------------------------------------------

        messages = []


        if request.system_message:

            messages.append({
                "role": "system",
                "content": request.system_message,
            })


        messages.append({
            "role": "user",
            "content": request.prompt,
        })


        # ----------------------------------------------------------
        # LLM inference
        # ----------------------------------------------------------

        outputs = pipe_llm(
            messages,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
        )


        # ----------------------------------------------------------
        # Extract assistant response
        # ----------------------------------------------------------

        generated_text = (
            outputs[0]["generated_text"][-1]["content"]
        )


        return {
            "generated_text": generated_text
        }


    except Exception as e:

        print(
            "LLM ERROR:",
            repr(e),
        )

        return {
            "error": str(e)
        }


# ==================================================================
# ASR
# ==================================================================

@app.post("/asr")
async def asr_endpoint(
    audio_file: UploadFile = File(...)
):

    tmp_file_path = None

    try:

        # ----------------------------------------------------------
        # Create temporary WAV file
        # ----------------------------------------------------------

        with tempfile.NamedTemporaryFile(
            suffix=".wav",
            delete=False,
        ) as tmp_file:

            audio_data = await audio_file.read()

            tmp_file.write(
                audio_data
            )

            tmp_file_path = tmp_file.name


        print(
            f"ASR input: {len(audio_data)} bytes"
        )


        # ----------------------------------------------------------
        # ASR inference
        # ----------------------------------------------------------

        with torch.inference_mode():

            result = stt_model_asr.transcribe(
                [tmp_file_path],
                batch_size=1,
            )


        # ----------------------------------------------------------
        # Extract transcription
        # ----------------------------------------------------------

        output = result[0]

        text = (
            output.text
            if hasattr(output, "text")
            else str(output)
        )


        text = text.strip()


        print(
            "ASR result:",
            text,
        )


        return {
            "transcribed_text": text
        }


    except Exception as e:

        print(
            "ASR ERROR:",
            repr(e),
        )

        return {
            "error": str(e)
        }


    finally:

        # ----------------------------------------------------------
        # Always remove temporary file
        # ----------------------------------------------------------

        if (
            tmp_file_path
            and os.path.exists(tmp_file_path)
        ):

            try:

                os.remove(
                    tmp_file_path
                )

            except Exception as cleanup_error:

                print(
                    "ASR cleanup error:",
                    repr(cleanup_error),
                )


# ==================================================================
# Ngrok Integration
# ==================================================================

print(
    "Setting up ngrok..."
)


# ==================================================================
# Allow asyncio inside Colab
# ==================================================================

nest_asyncio.apply()


# ==================================================================
# Ngrok Authentication
# ==================================================================

NGROK_AUTH_TOKEN = os.environ.get(
    "NGROK_AUTH_TOKEN"
)


if not NGROK_AUTH_TOKEN:

    NGROK_AUTH_TOKEN = input(
        "Enter your ngrok authentication token: "
    )


ngrok.set_auth_token(
    NGROK_AUTH_TOKEN
)


os.environ[
    "NGROK_AUTH_TOKEN"
] = NGROK_AUTH_TOKEN


# ==================================================================
# Server Configuration
# ==================================================================

PORT = 8002


# ==================================================================
# Start Uvicorn
# ==================================================================

def run_uvicorn():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=PORT,
        log_level="info",
    )


uvicorn_thread = threading.Thread(
    target=run_uvicorn,
    daemon=True,
)


uvicorn_thread.start()


# ==================================================================
# Wait for server
# ==================================================================

time.sleep(5)


# ==================================================================
# Connect Ngrok
# ==================================================================

try:

    tunnel = ngrok.connect(
        addr=PORT,
        proto="http",
    )


    public_url = tunnel.public_url


    print()
    print("=" * 70)
    print("Hindi AI Voice Assistant")
    print("=" * 70)

    print()
    print(
        "Frontend:"
    )

    print(
        f"{public_url}/"
    )

    print()
    print(
        "Status:"
    )

    print(
        f"{public_url}/status"
    )

    print()
    print(
        "Swagger:"
    )

    print(
        f"{public_url}/docs"
    )

    print()
    print(
        "TTS:"
    )

    print(
        f"{public_url}/tts"
    )

    print()
    print(
        "LLM:"
    )

    print(
        f"{public_url}/llm"
    )

    print()
    print(
        "ASR:"
    )

    print(
        f"{public_url}/asr"
    )

    print()
    print("=" * 70)


except Exception as e:

    print(
        "Error connecting to ngrok:"
    )

    print(
        repr(e)
    )

Setting up ngrok...
Enter your ngrok authentication token: 3HjKWm0efhpYTJumWO2avuENLGC_7yJenSQ4BvMJiiPSDRQ3F


INFO:     Started server process [10850]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8002 (Press CTRL+C to quit)



Hindi AI Voice Assistant

Frontend:
https://7148-34-182-66-228.ngrok-free.app/

Status:
https://7148-34-182-66-228.ngrok-free.app/status

Swagger:
https://7148-34-182-66-228.ngrok-free.app/docs

TTS:
https://7148-34-182-66-228.ngrok-free.app/tts

LLM:
https://7148-34-182-66-228.ngrok-free.app/llm

ASR:
https://7148-34-182-66-228.ngrok-free.app/asr



In [ ]:
"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Hindi AI Voice Assistant</title>

<style>
*{box-sizing:border-box}

body{
    margin:0;
    background:#090b0f;
    color:#f4f5f7;
    font-family:Inter,system-ui,-apple-system,Segoe UI,sans-serif
}

.app{
    max-width:760px;
    margin:auto;
    height:100vh;
    display:flex;
    flex-direction:column;
    padding:22px
}

header{
    display:flex;
    justify-content:space-between;
    align-items:center;
    padding:4px 0 18px
}

h1{
    font-size:20px;
    margin:0
}

.sub{
    font-size:12px;
    color:#858d99;
    margin-top:4px
}

.status{
    font-size:12px;
    color:#8d96a3;
    display:flex;
    gap:7px;
    align-items:center
}

.dot{
    width:7px;
    height:7px;
    border-radius:50%;
    background:#555
}

.online{
    background:#35d07f
}

.chat{
    flex:1;
    overflow:auto;
    border:1px solid #20252d;
    background:#0e1116;
    border-radius:16px;
    padding:18px
}

.msg{
    display:flex;
    margin:12px 0
}

.msg.user{
    justify-content:flex-end
}

.bubble{
    max-width:78%;
    padding:11px 14px;
    border-radius:15px;
    line-height:1.5;
    font-size:14px;
    white-space:pre-wrap
}

.ai .bubble{
    background:#191e26
}

.user .bubble{
    background:#6955e8
}

.typing{
    color:#89919d;
    font-size:13px;
    padding:10px 4px;
    display:none
}

.controls{
    padding-top:14px
}

.row{
    display:flex;
    gap:9px
}

input{
    flex:1;
    background:#11151b;
    border:1px solid #272e38;
    color:#fff;
    border-radius:11px;
    padding:12px;
    outline:none
}

button{
    border:0;
    border-radius:11px;
    padding:12px 17px;
    background:#6955e8;
    color:#fff;
    font-weight:600;
    cursor:pointer
}

button:hover{
    background:#5945d2
}

button:disabled{
    opacity:.45;
    cursor:not-allowed
}

.mic{
    width:52px;
    font-size:20px;
    padding:0
}

.mic.recording{
    background:#d94646
}

.voice-row{
    display:flex;
    justify-content:space-between;
    align-items:center;
    margin-top:9px;
    color:#737d8a;
    font-size:11px
}

select{
    background:#11151b;
    color:#ddd;
    border:1px solid #272e38;
    border-radius:7px;
    padding:5px
}

audio{
    display:none
}

@media(max-width:600px){
    .app{
        padding:12px
    }

    .bubble{
        max-width:88%
    }
}
</style>
</head>

<body>

<div class="app">

<header>
    <div>
        <h1>Hindi AI Assistant</h1>
        <div class="sub">Voice conversation</div>
    </div>

    <div class="status">
        <span id="dot" class="dot"></span>
        <span id="status">Connecting</span>
    </div>
</header>

<div id="chat" class="chat">
    <div class="msg ai">
        <div class="bubble">नमस्ते। मैं आपकी कैसे मदद कर सकता हूँ?</div>
    </div>
</div>

<div id="typing" class="typing">
    AI is thinking...
</div>

<div class="controls">

    <div class="row">
        <input
            id="text"
            placeholder="Type or press the microphone..."
            autocomplete="off"
        >

        <button id="send" onclick="sendText()">Send</button>

        <button
            id="mic"
            class="mic"
            onclick="toggleRecording()"
        >●</button>
    </div>

    <div class="voice-row">

        <span id="hint">
            Press the microphone and speak.
        </span>

        <label>
            Voice
            <select id="voice">
                <option value="female">Female</option>
                <option value="male">Male</option>
            </select>
        </label>

    </div>

</div>

<audio id="player"></audio>

</div>

<script>

/* ============================================================
   API
   ============================================================ */

const API = "https://0dc7-34-182-66-228.ngrok-free.app";


/* ============================================================
   State
   ============================================================ */

let busy = false;
let recording = false;

let stream = null;
let ctx = null;
let processor = null;
let source = null;

let chunks = [];

let speechStarted = false;
let silenceStart = 0;


/* ============================================================
   Helpers
   ============================================================ */

const $ = id => document.getElementById(id);


/* ============================================================
   API Status
   ============================================================ */

async function init(){

    try{

        const r = await fetch(
            API + "/status",
            {
                method: "GET",
                cache: "no-store"
            }
        );

        if(!r.ok){
            throw new Error("Status " + r.status);
        }

        $("dot").classList.add("online");
        $("status").textContent = "Online";

    }catch(e){

        $("status").textContent = "API offline";

        console.error(
            "API status error:",
            e
        );
    }
}


/* ============================================================
   Chat UI
   ============================================================ */

function addMsg(text, who){

    const m = document.createElement("div");
    m.className = "msg " + who;

    const b = document.createElement("div");
    b.className = "bubble";

    b.textContent = text;

    m.appendChild(b);

    $("chat").appendChild(m);

    $("chat").scrollTop =
        $("chat").scrollHeight;
}


function thinking(show){

    $("typing").style.display =
        show ? "block" : "none";

    $("chat").scrollTop =
        $("chat").scrollHeight;
}


function setBusy(value){

    busy = value;

    $("send").disabled = value;
    $("text").disabled = value;
}


/* ============================================================
   Text Input
   ============================================================ */

async function sendText(){

    if(busy){
        return;
    }

    const text =
        $("text").value.trim();

    if(!text){
        return;
    }

    $("text").value = "";

    addMsg(
        text,
        "user"
    );

    await pipeline(text);
}


$("text").addEventListener(
    "keydown",
    e => {

        if(e.key === "Enter"){

            e.preventDefault();

            sendText();
        }
    }
);


/* ============================================================
   LLM -> TTS Pipeline
   ============================================================ */

async function pipeline(text){

    setBusy(true);
    thinking(true);

    try{

        /* ----------------------------------------------------
           LLM
           ---------------------------------------------------- */

        const r = await fetch(
            API + "/llm",
            {
                method: "POST",

                headers: {
                    "Content-Type": "application/json"
                },

                body: JSON.stringify({

                    prompt: text,

                    system_message:
                        "You are a natural Hindi voice assistant. " +
                        "Reply conversationally and concisely in Hindi. " +
                        "Do not use markdown, emojis, or long explanations."
                })
            }
        );


        if(!r.ok){

            const errorText =
                await r.text();

            throw new Error(
                `LLM ${r.status}: ${errorText}`
            );
        }


        const d =
            await r.json();


        const reply =
            d.generated_text ??
            d.response ??
            d.text ??
            d.output ??
            "";


        if(!reply.trim()){

            throw new Error(
                "LLM returned an empty response"
            );
        }


        thinking(false);

        addMsg(
            reply,
            "ai"
        );


        /* ----------------------------------------------------
           TTS
           ---------------------------------------------------- */

        const t = await fetch(
            API + "/tts",
            {
                method: "POST",

                headers: {
                    "Content-Type": "application/json"
                },

                body: JSON.stringify({

                    text: reply,

                    voice:
                        $("voice").value
                })
            }
        );


        if(!t.ok){

            const errorText =
                await t.text();

            throw new Error(
                `TTS ${t.status}: ${errorText}`
            );
        }


        const blob =
            await t.blob();


        const url =
            URL.createObjectURL(blob);


        const player =
            $("player");


        player.src = url;


        await player.play();


        player.onended = () => {

            URL.revokeObjectURL(url);

        };


    }catch(e){

        thinking(false);

        addMsg(
            "Error: " + e.message,
            "ai"
        );

        console.error(
            "Pipeline error:",
            e
        );

    }finally{

        setBusy(false);

    }
}


/* ============================================================
   Recording
   ============================================================ */

async function toggleRecording(){

    if(recording){

        await stopRecording();

        return;
    }


    if(busy){

        return;
    }


    try{

        stream =
            await navigator.mediaDevices
                .getUserMedia({
                    audio: true
                });


        ctx =
            new AudioContext();


        const sampleRate =
            ctx.sampleRate;


        console.log(
            "Audio sample rate:",
            sampleRate
        );


        source =
            ctx.createMediaStreamSource(
                stream
            );


        /*
         * ScriptProcessorNode is deprecated but works
         * for this prototype.
         */

        processor =
            ctx.createScriptProcessor(
                4096,
                1,
                1
            );


        chunks = [];

        speechStarted = false;

        silenceStart = 0;


        processor.onaudioprocess = e => {

            const data =
                e.inputBuffer
                    .getChannelData(0);


            chunks.push(
                new Float32Array(data)
            );


            let sum = 0;


            for(
                let i = 0;
                i < data.length;
                i++
            ){

                sum +=
                    data[i] * data[i];
            }


            const rms =
                Math.sqrt(
                    sum / data.length
                );


            if(rms > 0.012){

                speechStarted = true;

                silenceStart =
                    performance.now();

            }
            else if(
                speechStarted &&
                performance.now() -
                silenceStart > 1200
            ){

                stopRecording();
            }

        };


        source.connect(
            processor
        );


        processor.connect(
            ctx.destination
        );


        recording = true;


        $("mic")
            .classList
            .add("recording");


        $("mic").textContent = "■";


        $("hint").textContent =
            "Listening... pause for 1.2s to send";


    }catch(e){

        console.error(
            "Microphone error:",
            e
        );

        addMsg(
            "Microphone permission is required.",
            "ai"
        );
    }
}


/* ============================================================
   Stop Recording
   ============================================================ */

async function stopRecording(){

    if(!recording){

        return;
    }


    recording = false;


    $("mic")
        .classList
        .remove("recording");


    $("mic").textContent = "●";


    $("hint").textContent =
        "Processing speech...";


    /*
     * IMPORTANT:
     * Save the actual browser sample rate
     * before closing AudioContext.
     */

    const sampleRate =
        ctx ? ctx.sampleRate : 44100;


    if(processor){

        processor.disconnect();

        processor = null;
    }


    if(source){

        source.disconnect();

        source = null;
    }


    if(stream){

        stream
            .getTracks()
            .forEach(
                track => track.stop()
            );

        stream = null;
    }


    if(ctx){

        await ctx.close();

        ctx = null;
    }


    if(!speechStarted){

        $("hint").textContent =
            "Press the microphone and speak.";

        chunks = [];

        return;
    }


    const wav =
        encodeWav(
            chunks,
            sampleRate
        );


    chunks = [];


    await sendAudio(
        wav
    );


    $("hint").textContent =
        "Press the microphone and speak.";
}


/* ============================================================
   WAV Encoder
   ============================================================ */

function encodeWav(
    buffers,
    sampleRate
){

    let len = 0;


    buffers.forEach(
        b => {
            len += b.length;
        }
    );


    const out =
        new Float32Array(len);


    let offset = 0;


    buffers.forEach(
        b => {

            out.set(
                b,
                offset
            );

            offset +=
                b.length;
        }
    );


    const buffer =
        new ArrayBuffer(
            44 + len * 2
        );


    const view =
        new DataView(buffer);


    const writeString =
        (offset, string) => {

            for(
                let i = 0;
                i < string.length;
                i++
            ){

                view.setUint8(
                    offset + i,
                    string.charCodeAt(i)
                );
            }
        };


    /* RIFF */

    writeString(
        0,
        "RIFF"
    );


    view.setUint32(
        4,
        36 + len * 2,
        true
    );


    writeString(
        8,
        "WAVE"
    );


    /* fmt */

    writeString(
        12,
        "fmt "
    );


    view.setUint32(
        16,
        16,
        true
    );


    /* PCM */

    view.setUint16(
        20,
        1,
        true
    );


    /* Mono */

    view.setUint16(
        22,
        1,
        true
    );


    /* Sample rate */

    view.setUint32(
        24,
        sampleRate,
        true
    );


    /* Byte rate */

    view.setUint32(
        28,
        sampleRate * 2,
        true
    );


    /* Block align */

    view.setUint16(
        32,
        2,
        true
    );


    /* Bits per sample */

    view.setUint16(
        34,
        16,
        true
    );


    /* data */

    writeString(
        36,
        "data"
    );


    view.setUint32(
        40,
        len * 2,
        true
    );


    /* PCM samples */

    for(
        let i = 0;
        i < len;
        i++
    ){

        let x =
            Math.max(
                -1,
                Math.min(
                    1,
                    out[i]
                )
            );


        view.setInt16(
            44 + i * 2,

            x < 0
                ? x * 32768
                : x * 32767,

            true
        );
    }


    return new Blob(
        [buffer],
        {
            type: "audio/wav"
        }
    );
}


/* ============================================================
   Send Audio -> ASR
   ============================================================ */

async function sendAudio(blob){

    if(busy){

        return;
    }


    setBusy(true);
    thinking(true);


    try{

        const form =
            new FormData();


        /*
         * IMPORTANT:
         * FastAPI expects:
         *
         * audio_file: UploadFile
         *
         * Therefore the field MUST be
         * named "audio_file".
         */

        form.append(
            "audio_file",
            blob,
            "speech.wav"
        );


        const r =
            await fetch(
                API + "/asr",
                {
                    method: "POST",
                    body: form
                }
            );


        if(!r.ok){

            const errorText =
                await r.text();


            throw new Error(
                `ASR ${r.status}: ${errorText}`
            );
        }


        const d =
            await r.json();


        /*
         * Backend returns:
         *
         * {
         *     "transcribed_text": "..."
         * }
         */

        const text =
            d.transcribed_text ??
            d.text ??
            d.transcription ??
            d.transcript ??
            d.response ??
            "";


        if(!text.trim()){

            throw new Error(
                "No speech detected"
            );
        }


        thinking(false);


        addMsg(
            text,
            "user"
        );


        /*
         * Send transcription to LLM
         * and then TTS.
         */

        await pipeline(
            text
        );


    }catch(e){

        thinking(false);


        addMsg(
            "Speech error: " +
            e.message,
            "ai"
        );


        console.error(
            "ASR error:",
            e
        );


    }finally{

        setBusy(false);
    }
}


/* ============================================================
   Start
   ============================================================ */

init();

</script>

</body>
</html>

